---
title: Automatic Piano Transcription Experiment
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.3
  kernelspec:
    display_name: Python 3
    language: python
    name: python3
---


This notebook is intentionally short. The implementation lives in editable Python modules under `src/piano_modeling/`, while realtime/deployment code lives under `src/piano_live_inference/`.


# Setup

In [1]:
#@title 1. Clone/update repo and install dependencies
import os

REPO_URL = "https://github.com/JDub323/piano_amt.git"  # change if this repo URL changes
REPO_DIR = "/content/piano_amt"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
%pip -q install -r requirements.txt
%pip -q install -e .

Cloning into '/content/piano_amt'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 124 (delta 45), reused 109 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 83.50 KiB | 3.21 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/piano_amt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 57.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piano_amt (pyproject.toml) ... done


In [2]:
#@title 2. Autoreload project modules
import sys
import types
import importlib
imp = types.ModuleType("imp")
imp.reload = importlib.reload
sys.modules["imp"] = imp
%reload_ext autoreload
%autoreload 2

Remember: autoreload reloads module definitions, not objects. Recreate the `system`, loaders, optimizer, or config after changing their definitions.

In [4]:
#@title 3. Imports and runtime setup
from dataclasses import asdict
import gc
import json
from pathlib import Path

import pandas as pd
import torch

sys.path.insert(0, "/content/piano_amt/src")

from piano_modeling import Config, config_to_json, setup_runtime, make_colab_paths, print_paths
from piano_modeling.maestro_cache import load_cached_maestro_manifest, preprocess_maestro_in_download_batches
from piano_modeling.sliced_cache import load_or_build_sliced_dataset
from piano_modeling.models import PianoTranscriptionSystem, count_parameters_millions
from piano_modeling.training import run_training, load_checkpoint
from piano_modeling.benchmark import describe_runtime, sweep_training_settings, apply_best_benchmark_config
from piano_modeling.diagnostics import check_dataset_artifacts
from piano_modeling.export import export_prediction_bundle

DEVICE = setup_runtime(seed=1337)
describe_runtime(DEVICE)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 39.5 GB
CUDA: 12.8
PyTorch: 2.11.0+cu128


In [13]:
#@title 4. Config
cfg = Config()

# Optional per-run overrides can still live here:
# cfg.resnet_name = "resnet34"
# cfg.batch_size = 32
# cfg.num_workers = 2
# cfg.lr = 2e-4
# cfg.epochs = 40
cfg.compile_model = False

print(config_to_json(cfg))

{
  "maestro_version": "v3.0.0",
  "midi_min": 21,
  "midi_max": 108,
  "sample_rate": 16000,
  "segment_seconds": 12.0,
  "hop_length": 160,
  "n_fft": 2048,
  "n_mels": 229,
  "f_min": 27.5,
  "f_max": 8000.0,
  "feature_type": "mel",
  "cqt_bins_per_octave": 48,
  "cqt_n_bins": 384,
  "resnet_name": "resnet34.a1_in1k",
  "pretrained": true,
  "decoder_channels": 256,
  "decoder_type": "fpn",
  "dropout": 0.1,
  "compile_model": false,
  "FIND_OPTIMAL_SETTINGS": false,
  "batch_size": 32,
  "num_workers": 2,
  "epochs": 40,
  "lr": 0.0002,
  "weight_decay": 0.0001,
  "grad_clip": 1.0,
  "use_amp": true,
  "onset_loss_weight": 2.0,
  "offset_loss_weight": 1.0,
  "frame_loss_weight": 1.0,
  "velocity_loss_weight": 0.5,
  "sustain_loss_weight": 0.5,
  "onset_pos_weight": 16.0,
  "offset_pos_weight": 12.0,
  "frame_pos_weight": 3.0,
  "sustain_pos_weight": 2.0,
  "onset_width_frames": 2,
  "offset_width_frames": 2,
  "sustain_threshold": 64,
  "enable_audio_augment": false,
  "enable_spe

In [6]:
#@title 5. Mount Drive, create paths, and define cache-safety flags
from google.colab import drive

drive.mount('/content/drive')

paths = make_colab_paths('/content/drive/MyDrive/piano_transcription_resnet')
print_paths(paths)

# Safe defaults: use the existing pre-sliced dataset and do not redo expensive work.
USE_PRE_SLICED_DATASET = True
ALLOW_REBUILD_IF_SLICED_ZIP_MISSING = False
FORCE_RESPLICE_SLICED_CACHE = False

# Full-song MAESTRO preprocessing is fallback-only. Keep False unless intentionally rebuilding.
RUN_MAESTRO_PREPROCESS = False
NUM_MAESTRO_DOWNLOAD_BATCHES = 3
START_BATCH_INDEX = 0
END_BATCH_INDEX = -1
OVERWRITE_EXISTING_SPECS = False

Mounted at /content/drive
project_root: /content/drive/MyDrive/piano_transcription_resnet
datasets_root: /content/drive/MyDrive/piano_transcription_resnet/datasets
maestro_meta_root: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_metadata
maestro_work_root: /content/maestro_working_audio
maestro_spec_root: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_spectrogram_tensors
maestro_midi_root: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_midi
maestro_cache_manifest: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_spectrogram_tensors/maestro-v3.0.0_cached_manifest.csv
sliced_zip_path: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_sliced.zip
sliced_root: /content/local_maestro_sliced
sliced_manifest: /content/local_maestro_sliced/sliced_manifest.csv
checkpoint_dir: /content/drive/MyDrive/piano_transcription_resnet/checkpoints
export_dir: /content/drive/MyD

In [7]:
#@title 6. Load metadata/full-song manifest without long downloads
# This downloads tiny MAESTRO metadata only if missing. It does not download the full audio dataset.
meta = load_cached_maestro_manifest(paths, cfg, download_if_missing=True)

if RUN_MAESTRO_PREPROCESS:
    meta = preprocess_maestro_in_download_batches(
        paths,
        cfg,
        num_batches=NUM_MAESTRO_DOWNLOAD_BATCHES,
        start_batch=START_BATCH_INDEX,
        end_batch=None if END_BATCH_INDEX < 0 else END_BATCH_INDEX,
        overwrite_existing_specs=OVERWRITE_EXISTING_SPECS,
        device=DEVICE,
    )
else:
    print('Full-song spectrogram preprocessing is disabled. The training path will restore/use the pre-sliced zip instead.')

metadata_csv: /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_metadata/maestro-v3.0.0.csv True
   canonical_composer                canonical_title       split  year  \
0          Alban Berg                   Sonata Op. 1       train  2018   
1          Alban Berg                   Sonata Op. 1       train  2008   
2          Alban Berg                   Sonata Op. 1       train  2017   
3  Alexander Scriabin  24 Preludes Op. 11, No. 13-24       train  2004   
4  Alexander Scriabin               3 Etudes, Op. 65  validation  2006   

                                       midi_filename  \
0  2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...   
1  2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...   
2  2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...   
3  2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...   
4  2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...   

                                      audio_filename    duration  \
0  2018/MIDI-Unprocessed_Cham

In [8]:
#@title 7. Restore/load pre-sliced MAESTRO dataset
# Normal path: if /content/local_maestro_sliced exists, reuse it.
# Otherwise restore paths.sliced_zip_path from Drive.
# It only rebuilds/reslices if you explicitly enable the flags above.
sliced_meta = load_or_build_sliced_dataset(
    paths,
    cfg,
    meta_df=meta,
    use_pre_sliced_dataset=USE_PRE_SLICED_DATASET,
    allow_rebuild_if_sliced_zip_missing=ALLOW_REBUILD_IF_SLICED_ZIP_MISSING,
    force_resplice_sliced_cache=FORCE_RESPLICE_SLICED_CACHE,
)

Restoring pre-sliced dataset from Drive zip:
  /content/drive/MyDrive/piano_transcription_resnet/datasets/maestro-v3.0.0_sliced.zip
-> /content/local_maestro_sliced
Restored sliced dataset to /content/local_maestro_sliced
Loading sliced manifest: /content/local_maestro_sliced/sliced_manifest.csv
Compatibility check passed by tensor shape for 9 sampled chunks. 9 sampled chunks were legacy files without metadata.
Sliced chunk counts:
split
train         47420
test           5935
validation     5782
Name: count, dtype: int64


In [14]:
#@title 8. Build model
system = PianoTranscriptionSystem(cfg).to(DEVICE)

if cfg.compile_model:
    system = torch.compile(system)

print('Parameters:', count_parameters_millions(system), 'M')

Parameters: 27.802885 M


In [17]:
#@title 9. Optional benchmark batch size/workers
RUN_BENCHMARK = True  #@param {type:'boolean'}

torch.cuda.empty_cache()

if RUN_BENCHMARK:
    bench_df = sweep_training_settings(
        sliced_meta,
        base_cfg=cfg,
        batch_sizes=(16, 24, 32),
        num_workers_list=(2, 4, 8),
        amp_options=(True,),
        device=DEVICE,
    )
    display(bench_df)
    cfg = apply_best_benchmark_config(cfg, bench_df, memory_soft_limit_gb=14.0)
else:
    print('Skipping benchmark.')


Testing batch=16, workers=2, amp=True
Training chunks per epoch: 1024
Validation chunks: 1
{'batch_size': 16, 'num_workers': 2, 'use_amp': True, 'grad_accum_steps': 1, 'effective_batch_size': 16, 'lr': 0.0002, 'compile_model': False, 'status': 'ok', 'chunks_per_sec': 14.000110171023586, 'optimizer_steps_per_sec': 0.8750068856889741, 'sec_per_optimizer_step': 1.14284814937497, 'peak_gpu_gb': 24.692862033843994}

Testing batch=16, workers=4, amp=True
Training chunks per epoch: 1024
Validation chunks: 1
{'batch_size': 16, 'num_workers': 4, 'use_amp': True, 'grad_accum_steps': 1, 'effective_batch_size': 16, 'lr': 0.0002, 'compile_model': False, 'status': 'ok', 'chunks_per_sec': 13.9870818291038, 'optimizer_steps_per_sec': 0.8741926143189875, 'sec_per_optimizer_step': 1.1439126613749977, 'peak_gpu_gb': 24.692862033843994}

Testing batch=16, workers=8, amp=True
Training chunks per epoch: 1024
Validation chunks: 1
{'batch_size': 16, 'num_workers': 8, 'use_amp': True, 'grad_accum_steps': 1, '

,batch_size,num_workers,use_amp,grad_accum_steps,effective_batch_size,lr,compile_model,status,chunks_per_sec,optimizer_steps_per_sec,sec_per_optimizer_step,peak_gpu_gb
0,24,2,True,1,24,0.0002,False,OOM,0.000000,0.000000,NaN,NaN
1,24,4,True,1,24,0.0002,False,OOM,0.000000,0.000000,NaN,NaN
2,24,8,True,1,24,0.0002,False,OOM,0.000000,0.000000,NaN,NaN
3,32,2,True,1,32,0.0002,False,OOM,0.000000,0.000000,NaN,NaN
4,32,4,True,1,32,0.0002,False,OOM,0.000000,0.000000,NaN,NaN
5,32,8,True,1,32,0.0002,False,OOM,0.000000,0.000000,NaN,NaN
6,16,8,True,1,16,0.0002,False,ok,14.006353,0.875397,1.142339,24.690511
7,16,2,True,1,16,0.0002,False,ok,14.000110,0.875007,1.142848,24.692862
8,16,4,True,1,16,0.0002,False,ok,13.987082,0.874193,1.143913,24.692862


Chosen config:
cfg.batch_size = 16
cfg.num_workers = 2
cfg.use_amp = True
batch_size                        16
num_workers                        2
use_amp                         True
grad_accum_steps                   1
effective_batch_size              16
lr                            0.0002
compile_model                  False
status                            ok
chunks_per_sec              14.00011
optimizer_steps_per_sec     0.875007
sec_per_optimizer_step      1.142848
peak_gpu_gb                24.692862
speed_ratio                 0.999554
Name: 7, dtype: object


In [ ]:
#@title 10. Start or resume training
RUN_TRAINING = True    #@param {type:'boolean'}
CUSTOM_RUN_SUFFIX = "" #@param {type:'string'}
TRAIN_SAMPLES_PER_EPOCH = 20000 #@param {type:'integer'}
VAL_SAMPLES = 0 #@param {type:'integer'}
RESUME = False #@param {type:'boolean'}

torch.cuda.empty_cache()

if RUN_TRAINING:
    train_result = run_training(
        system=system,
        cfg=cfg,
        paths=paths,
        sliced_meta=sliced_meta,
        custom_run_suffix=CUSTOM_RUN_SUFFIX,
        train_samples_per_epoch=TRAIN_SAMPLES_PER_EPOCH,
        val_samples=VAL_SAMPLES,
        resume=RESUME,
        device=DEVICE,
    )
    RUN_NAME = train_result['run_name']
    best_ckpt = train_result['best_ckpt']
    last_ckpt = train_result['last_ckpt']
else:
    train_result = None
    RUN_NAME = None
    best_ckpt = None
    last_ckpt = None
    print('Skipping training.')

Training chunks per epoch: 20000
Validation chunks: 5782
Run directory: /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12
RUN_NAME: resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12
TensorBoard log directory: /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12/tensorboard


train epoch 1:   0%|          | 0/1250 [00:00<?, ?it/s]


[GPU Memory - Batch 0] Allocated: 0.15 GB, Reserved: 5.85 GB


validate:   0%|          | 0/362 [00:00<?, ?it/s]

Epoch 1: new best metric=0.7561; saved /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12/best.pt
{'epoch': 1, 'train_onset': 0.3467340646624565, 'train_offset': 0.1949645089328289, 'train_frame': 0.25716348485946655, 'train_sustain': 0.25782342755794524, 'train_velocity': 0.006059446956962347, 'train_total': 1.0627449326515197, 'val_onset': 0.17327858378775018, 'val_offset': 0.10987669786174831, 'val_frame': 0.17051748749460163, 'val_sustain': 0.19605592107616135, 'val_velocity': 0.0035230600631392175, 'val_total': 0.6532517488709538, 'val_onset_precision': 0.2013759740069026, 'val_onset_recall': 0.4925344569575107, 'val_onset_f1': 0.28587149457871797, 'val_offset_precision': 0.08726904094818315, 'val_offset_recall': 0.19156607355594177, 'val_offset_f1': 0.119911637005739, 'val_frame_precision': 0.311421862393434, 'val_frame_recall': 0.40039254110654454, 'val_frame_f1': 0.35034691691187014, 'val_sustain_precision': 0.

...


[GPU Memory - Batch 0] Allocated: 0.47 GB, Reserved: 7.21 GB


validate:   0%|          | 0/362 [00:00<?, ?it/s]

Epoch 2: new best metric=1.0877; saved /content/drive/MyDrive/piano_transcription_resnet/checkpoints/resnet34.a1_in1k_maestro_v3.0.0_mel_sr16000_hop160_seg12/best.pt
{'epoch': 2, 'train_onset': 0.16346900889277458, 'train_offset': 0.10180888929367066, 'train_frame': 0.1414901226937771, 'train_sustain': 0.20178472527265548, 'train_velocity': 0.0042731582790613174, 'train_total': 0.6128259045362473, 'val_onset': 0.11249940134185443, 'val_offset': 0.08166127257366075, 'val_frame': 0.11598331566007028, 'val_sustain': 0.20506445897033743, 'val_velocity': 0.0028529803614756747, 'val_total': 0.5180614297054335, 'val_onset_precision': 0.2637676186142807, 'val_onset_recall': 0.7270740503428654, 'val_onset_f1': 0.38710239349748554, 'val_offset_precision': 0.11019286723494263, 'val_offset_recall': 0.4867797282346689, 'val_offset_f1': 0.17970558237712206, 'val_frame_precision': 0.42955041225343693, 'val_frame_recall': 0.6615568365648032, 'val_frame_f1': 0.5208874053091898, 'val_sustain_precision':

train epoch 3:   0%|          | 0/1250 [00:00<?, ?it/s]


[GPU Memory - Batch 0] Allocated: 0.47 GB, Reserved: 27.12 GB


# Inference/export examples

In [ ]:
#@title 11. Export a prediction bundle for an audio file
RUN_EXPORT_EXAMPLE = True #@param {type:'boolean'}
AUDIO_PATH = "/path/to/piano.wav" #@param {type:'string'}
OUT_STEM = "my_transcription" #@param {type:'string'}
CHECKPOINT_TO_LOAD = "best" #@param ["best", "last", "custom"]
CUSTOM_CHECKPOINT_PATH = "" #@param {type:'string'}

if RUN_EXPORT_EXAMPLE:
    if CHECKPOINT_TO_LOAD == 'custom':
        ckpt_path = Path(CUSTOM_CHECKPOINT_PATH)
    elif CHECKPOINT_TO_LOAD == 'last':
        ckpt_path = last_ckpt
    else:
        ckpt_path = best_ckpt
    load_checkpoint(ckpt_path, system, map_location=DEVICE, cfg=cfg)
    result = export_prediction_bundle(AUDIO_PATH, system, OUT_STEM, paths.export_dir, cfg)
    result
else:
    print('Skipping export example.')

# Diagnostics / optional live inference

In [ ]:
#@title 12. Check expected Drive/local artifacts
RUN_ARTIFACT_CHECK = False #@param {type:'boolean'}
if RUN_ARTIFACT_CHECK:
    check_dataset_artifacts(paths)
else:
    print('Skipping artifact check.')

In [ ]:
#@title 13. Optional realtime imports
# The realtime code is separated so you can ignore it until the model is good.
# from piano_live_inference import RealTimePianoTranscriber, benchmark_latency
# from piano_live_inference.gradio_app import launch_gradio_realtime
# from piano_live_inference.deploy import export_torchscript
#
# load_checkpoint(best_ckpt, system, map_location=DEVICE, cfg=cfg)
# benchmark_latency(system, cfg, seconds=2.0, repeats=20, device=DEVICE)
# launch_gradio_realtime(system, cfg)